# 01 · Leakage audit (Level 1)

**Question:** how much of the reported performance comes from *when we stopped collecting data* rather than *what the data says*?

In the original pipeline (`GRIDS_FeatureEng_BERT_train.ipynb`, cell 16):

| | feature cutoff |
|---|---|
| patients who **had** respiratory failure | `rf_time` (the moment of failure) |
| patients who **did not** | `icu_intime + 12h` |

So `feature_window_hours` is computed *from the label*. Every negative has exactly 12.0 h; most positives have less.
This notebook measures that directly using only the saved `train_df.csv` / `test_df.csv` — no re-extraction needed.

**Input:** `train_df.csv`, `test_df.csv` (the files ClinicalBERT was trained on).
**Output:** a printed results table + `leakage_audit_results.json` and `leakage_audit_auroc.png` (aggregate numbers only — safe to commit).

⚠️ Do not add cells that print individual patient rows (`df.head()`, example texts). Notebook outputs get committed to GitHub, and MIMIC data may not be published.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ✏️ EDIT THIS: the folder that contains train_df.csv and test_df.csv
#   Colab + Drive : "/content/drive/MyDrive/USC/ICU-MM-main/outputs"
#   Kaggle        : "/kaggle/input/grids-newdatasplits"
#   Local PC      : r"C:\Users\<you>\Downloads\grids-newdatasplits"
DATA_DIR = Path("/content/drive/MyDrive/USC/ICU-MM-main/outputs")

# Where to write the (aggregate-only) results
OUT_DIR = Path("/content/drive/MyDrive/USC/icu-leakage-audit") if IN_COLAB else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL = "respiratory_failure"
SEED = 42

In [ ]:
# ── Load ──────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train_df.csv")
test_df  = pd.read_csv(DATA_DIR / "test_df.csv")

for name, df in [("train", train_df), ("test", test_df)]:
    print(f"{name:5s}: {len(df):>7,} stays | positive rate {df[LABEL].mean()*100:.1f}%")

assert "feature_window_hours" in train_df.columns, "This isn't the original train_df — feature_window_hours is missing."
y_tr = train_df[LABEL].values
y_te = test_df[LABEL].values

## Test 1 — Does the window length alone predict the label?

No model, no labs. Just one number per patient.
(Direction — e.g. "shorter window = higher risk" — is chosen on **train**, then scored on **test**.)

In [ ]:
def single_feature_auc(x_tr, x_te):
    """AUROC on test of a single raw column, with its sign chosen on train."""
    x_tr = pd.Series(x_tr).fillna(pd.Series(x_tr).median()).values
    x_te = pd.Series(x_te).fillna(pd.Series(x_tr).median()).values
    sign = 1 if roc_auc_score(y_tr, x_tr) >= 0.5 else -1
    return roc_auc_score(y_te, sign * x_te), sign

print("How the window is distributed:")
for lbl, name in [(0, "stable (neg)"), (1, "failure (pos)")]:
    w = train_df.loc[train_df[LABEL] == lbl, "feature_window_hours"]
    print(f"  {name:14s} median {w.median():5.2f} h | "
          f"exactly 12h: {(w.round(3) == 12).mean()*100:5.1f}% | "
          f"< 12h: {(w < 12 - 1e-6).mean()*100:5.1f}%")

count_cols = [c for c in train_df.columns if c.endswith("_count") and c != "iv_count" and c != "iv_drip_count"]
single = {}
single["feature_window_hours"], _ = single_feature_auc(train_df["feature_window_hours"], test_df["feature_window_hours"])

# Also worth checking: stay_duration_hours is the WHOLE ICU stay length — only known at discharge.
if "stay_duration_hours" in train_df.columns:
    single["stay_duration_hours (future info)"], _ = single_feature_auc(train_df["stay_duration_hours"], test_df["stay_duration_hours"])

# Quieter versions of the same leak: how MUCH data was collected
single["total lab draws (sum of *_count)"], _ = single_feature_auc(train_df[count_cols].sum(axis=1), test_df[count_cols].sum(axis=1))
if "total_presc" in train_df.columns:
    single["total_presc"], _ = single_feature_auc(train_df["total_presc"], test_df["total_presc"])
lab_value_cols = [c for c in train_df.columns if c.endswith("_mean") and not c.endswith("_early_mean")]
single["# labs never measured"], _ = single_feature_auc(train_df[lab_value_cols].isna().sum(axis=1), test_df[lab_value_cols].isna().sum(axis=1))
if "clinical_text" in train_df.columns:
    single["clinical_text word count"], _ = single_feature_auc(
        train_df["clinical_text"].fillna("").str.split().str.len(),
        test_df["clinical_text"].fillna("").str.split().str.len())

print("\nTest-set AUROC from ONE number, no model:")
for k, v in single.items():
    print(f"  {k:38s} {v:.4f}")

## Test 2 — Structured model, with and without the window-dependent features

| Set | What's included |
|---|---|
| **A. as-built** | every feature the team used |
| **B. − window columns** | drop `feature_window_hours`, `obs_window_hours`, `is_short_stay`, `stay_duration_hours` |
| **C. − window + counts** | B, and also drop every `*_count`, `total_presc`, `unique_drugs`, `iv_count`, `iv_drip_count` |

Even **C** is an *upper bound*, not the honest number: which labs are missing, and the min/max/delta of labs, still depend on how long the window was.
The real number comes from `02_landmark_redesign.ipynb`.

In [ ]:
ID_COLS = {"stay_id", "subject_id", "hadm_id", LABEL, "clinical_text"}
all_feats = [c for c in train_df.columns
             if c not in ID_COLS and pd.api.types.is_numeric_dtype(train_df[c])]

WINDOW_COLS = ["feature_window_hours", "obs_window_hours", "is_short_stay", "stay_duration_hours"]
COUNT_COLS  = [c for c in all_feats if c.endswith("_count")] + \
              [c for c in ["total_presc", "unique_drugs", "iv_count", "iv_drip_count"] if c in all_feats]

feature_sets = {
    "A. as-built":            all_feats,
    "B. - window columns":    [c for c in all_feats if c not in WINDOW_COLS],
    "C. - window + counts":   [c for c in all_feats if c not in WINDOW_COLS and c not in COUNT_COLS],
}

def make_models():
    return {
        "LogReg": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                LogisticRegression(max_iter=3000, C=1.0)),
        "GradBoost": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED),
    }

rows, fitted = [], {}
for set_name, cols in feature_sets.items():
    for model_name, model in make_models().items():
        model.fit(train_df[cols], y_tr)
        p = model.predict_proba(test_df[cols])[:, 1]
        rows.append({"feature_set": set_name, "model": model_name, "n_features": len(cols),
                     "test_AUROC": roc_auc_score(y_te, p), "test_AUPRC": average_precision_score(y_te, p)})
        fitted[(set_name, model_name)] = (model, cols)
        print(f"{set_name:24s} {model_name:9s} ({len(cols):3d} feats)  AUROC {rows[-1]['test_AUROC']:.4f}  AUPRC {rows[-1]['test_AUPRC']:.4f}")

results = pd.DataFrame(rows)

## Test 3 — "Is it a timer?"

Take the **stable** test patients (who really had 12 h of data) and change only the window columns to pretend it's hour 2.
A real risk model shouldn't care much. A timer will panic.

In [ ]:
model, cols = fitted[("A. as-built", "GradBoost")]
stable = test_df[test_df[LABEL] == 0].copy()
p_true = model.predict_proba(stable[cols])[:, 1].mean()

fake = stable.copy()
fake["feature_window_hours"] = 2.0
if "obs_window_hours" in fake: fake["obs_window_hours"] = 2.0
p_fake = model.predict_proba(fake[cols])[:, 1].mean()

print(f"Stable patients, true 12h window  → mean predicted risk {p_true*100:5.1f}%")
print(f"Same patients, window set to 2h   → mean predicted risk {p_fake*100:5.1f}%")
print("(Only the window number changed — every lab and medication value is identical.)")

In [ ]:
# ── Save aggregate results (safe to commit: no patient-level data) ──
import matplotlib.pyplot as plt

summary = {
    "n_train": int(len(train_df)), "n_test": int(len(test_df)),
    "train_positive_rate": float(y_tr.mean()),
    "single_feature_test_auroc": {k: round(float(v), 4) for k, v in single.items()},
    "models": results.round(4).to_dict(orient="records"),
    "timer_test_mean_risk_stable": {"true_window": round(float(p_true), 4), "window_set_to_2h": round(float(p_fake), 4)},
}
with open(OUT_DIR / "leakage_audit_results.json", "w") as f:
    json.dump(summary, f, indent=2)

fig, ax = plt.subplots(figsize=(8, 4.5))
labels = ["window length\nalone"] + [f"{r.feature_set}\n{r.model}" for r in results.itertuples()]
vals   = [single["feature_window_hours"]] + list(results["test_AUROC"])
ax.barh(labels[::-1], vals[::-1], color=["#4C78A8"] * len(results) + ["#E45756"])
ax.set_xlim(0.5, 1.0); ax.set_xlabel("Test AUROC")
ax.set_title("How much of the score is the window length?")
for i, v in enumerate(vals[::-1]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout(); plt.savefig(OUT_DIR / "leakage_audit_auroc.png", dpi=150); plt.show()
print(f"Saved → {OUT_DIR}")